In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("Data/PBC_dataset_normal_DIB")

rows = []
for class_folder in sorted(data_dir.iterdir()):
    if class_folder.is_dir():
        for img_path in class_folder.glob("*.jpg"):
            if not img_path.name.startswith("."):
                rows.append({"path": str(img_path), "label": class_folder.name})

df = pd.DataFrame(rows)
print(df.shape)
print(df["label"].value_counts())
df.head()

(17092, 2)
label
neutrophil      3329
eosinophil      3117
ig              2895
platelet        2348
erythroblast    1551
monocyte        1420
basophil        1218
lymphocyte      1214
Name: count, dtype: int64


,path,label
0,Data\PBC_dataset_normal_DIB\basophil\BA_100102...,basophil
1,Data\PBC_dataset_normal_DIB\basophil\BA_101381...,basophil
2,Data\PBC_dataset_normal_DIB\basophil\BA_101611...,basophil
3,Data\PBC_dataset_normal_DIB\basophil\BA_102039...,basophil
4,Data\PBC_dataset_normal_DIB\basophil\BA_102750...,basophil


In [3]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

train: 11964 val: 2564 test: 2564


In [5]:
import pandas as pd

proportions = pd.DataFrame({
    "all": df["label"].value_counts(normalize=True),
    "train": train_df["label"].value_counts(normalize=True),
    "val": val_df["label"].value_counts(normalize=True),
    "test": test_df["label"].value_counts(normalize=True),
}).round(3)
proportions

,all,train,val,test
label,,,,
neutrophil,0.195,0.195,0.195,0.195
eosinophil,0.182,0.182,0.183,0.182
ig,0.169,0.169,0.170,0.169
platelet,0.137,0.137,0.138,0.137
erythroblast,0.091,0.091,0.090,0.091
monocyte,0.083,0.083,0.083,0.083
basophil,0.071,0.071,0.071,0.071
lymphocyte,0.071,0.071,0.071,0.071


In [6]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)
print("saved")

saved


In [7]:
from PIL import Image
import numpy as np

IMG_SIZE = (224, 224)

def load_image(path):
    img = Image.open(path).convert("RGB")
    img = img.resize(IMG_SIZE)
    return np.array(img) / 255.0

x = load_image(train_df.iloc[0]["path"])
print(x.shape, x.min(), x.max())

(224, 224, 3) 0.027450980392156862 1.0


In [8]:
bad = []
for p in df["path"]:
    try:
        load_image(p)
    except Exception:
        bad.append(p)
print("images checked:", len(df), "| unreadable:", len(bad))

images checked: 17092 | unreadable: 0
